## Zadanie domowe - Algorytm Canny'ego

Celem zadania domowego jest wykonanie pełnej implementacji algorytmu Canny'ego.

W ramach ćwiczenia w trakcie laboratorium wyznaczono obrazy $g_{NH}$ i $g_{NL}$.
Dla przypomnienia:
Można powiedzieć, że na obrazie $g_{NH}$ są "pewne" krawędzie.
Natomiast na $g_{NL}$ "potencjalne".
Często krawędzie "pewne" nie są ciągłe.
Wykorzystuje się więc krawędzie "potencjalne", aby uzupełnić nieciągłości.
Procedura wygląda następująco:
1. Stwórz stos zawierający wszystkie piksele zaznaczone na obrazie $g_{NH}$.
W tym celu wykorzystaj listę współrzędnych `[row, col]`.
Do pobrania elementu z początku służy metoda `list.pop()`.
Do dodania elementu na koniec listy służy metoda `list.append(new)`.
2. Stwórz obraz, który będzie zawierał informację czy dany piksel został już odwiedzony.
3. Stwórz obraz, który zawierać będzie wynikowe krawędzie.
Jej rozmiar jest równy rozmiarowi obrazu.
4. Wykonaj pętlę, która będzie pobierać elementy z listy, dopóki ta nie będzie pusta.
W tym celu najlepiej sprawdzi się pętla `while`.
    - W każdej iteracji pobierz element ze stosu.
    - Sprawdź, czy dany element został już odwiedzony.
    - Jeśli nie został, to:
        - Oznacz go jako odwiedzony,
        - Oznacz piksel jako krawędź w wyniku,
        - Sprawdź otoczenie piksela w obrazie $g_{NL}$,
        - Dodaj do stosu współrzędne otoczenia, które zawierają krawędź.
        Można to wykonać np. pętlą po stworzonym otoczeniu.
7. Wyświetl obraz oryginalny, obraz $g_{NH}$ oraz obraz wynikowy.
8. Porównaj wynik algorytmu z wynikiem OpenCV.

Pomocnicze obrazy $g_{NH}$ i $g_{NL}$ zostały wprowadzone dla uproszczenia opisu.
Algorytm można zaimplementować w bardziej "zwarty" sposób.

Na podstawie powyższego opisu zaimplementuj pełny algorytm Canny'ego.

In [ ]:
import cv2
from matplotlib import pyplot as plt
import numpy as np
import math
import os

if not os.path.exists("dom.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/09_Canny/dom.png --no-check-certificate

def quantize_angles(alpha_deg):
    h, w = alpha_deg.shape
    directions = np.zeros((h, w), dtype=np.uint8)

    for y in range(h):
        for x in range(w):
            a = alpha_deg[y, x]

            # d2 - poziomy (porownanie lewo/prawo)
            if (-22.5 <= a < 22.5) or (a >= 157.5) or (a < -157.5):
                directions[y, x] = 2

            # d4 - skos /
            elif (22.5 <= a < 67.5) or (-157.5 <= a < -112.5):
                directions[y, x] = 4

            # d1 - pionowy (porownanie gora/dol)
            elif (67.5 <= a < 112.5) or (-112.5 <= a < -67.5):
                directions[y, x] = 1

            # d3 - skos \
            else:
                directions[y, x] = 3

    return directions


def nonmax(directions, M):
    h, w = M.shape
    gN = np.zeros((h, w), dtype=np.float32)

    for y in range(1, h - 1):
        for x in range(1, w - 1):
            d = directions[y, x]
            m = M[y, x]

            if d == 1:
                n1 = M[y - 1, x]
                n2 = M[y + 1, x]
            elif d == 2:
                n1 = M[y, x - 1]
                n2 = M[y, x + 1]
            elif d == 3:
                n1 = M[y - 1, x - 1]
                n2 = M[y + 1, x + 1]
            else:  # d == 4
                n1 = M[y - 1, x + 1]
                n2 = M[y + 1, x - 1]

            if m >= n1 and m >= n2:
                gN[y, x] = m
            else:
                gN[y, x] = 0

    return gN

def hysteresis(gNH, gNL):
    h, w = gNH.shape

    stack = []
    for y in range(h):
        for x in range(w):
            if gNH[y, x]:
                stack.append((y, x))

    visited = np.zeros((h, w), dtype=bool)
    out = np.zeros((h, w), dtype=np.uint8)

    while stack:
        y, x = stack.pop()   

        if visited[y, x]:
            continue

        visited[y, x] = True
        out[y, x] = 255

        for ny in range(max(0, y - 1), min(h, y + 2)):
            for nx in range(max(0, x - 1), min(w, x + 2)):
                if not visited[ny, nx] and gNL[ny, nx]:
                    stack.append((ny, nx))

    return out



def canny_stage1(img, gauss_ksize=(5, 5), gauss_sigma=1.0, TL=20, TH=40):
    if img is None:
        raise ValueError("Obraz wejsciowy jest pusty")
    if TL >= TH:
        raise ValueError("TL musi byc mniejsze od TH")

    blur = cv2.GaussianBlur(img, gauss_ksize, gauss_sigma)

    gx = cv2.Sobel(blur, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(blur, cv2.CV_32F, 0, 1, ksize=3)

    M = np.hypot(gx, gy)
    alpha = np.arctan2(gy, gx)
    alpha_deg = np.rad2deg(alpha)

    directions = quantize_angles(alpha_deg)

    gN = nonmax(directions, M)

    gN_norm = cv2.normalize(gN, None, 0, 255, cv2.NORM_MINMAX)
    gN_norm = gN_norm.astype(np.uint8)

    gNH = gN_norm >= TH
    gNL = (gN_norm >= TL) & (gN_norm < TH)

    h, w = img.shape
    vis = np.zeros((h, w, 3), dtype=np.uint8)  
    vis[gNL] = [0, 0, 255]      
    vis[gNH] = [255, 0, 0]      


    return {
        "blur": blur,
        "gx": gx,
        "gy": gy,
        "M": M,
        "alpha_deg": alpha_deg,
        "directions": directions,
        "gN": gN,
        "gN_norm": gN_norm,
        "gNH": gNH,
        "gNL": gNL,
        "vis": vis
    }

def canny_full(img, gauss_ksize=(7, 7), gauss_sigma=1.4, TL=30, TH=90):
    stage1 = canny_stage1(
        img,
        gauss_ksize=gauss_ksize,
        gauss_sigma=gauss_sigma,
        TL=TL,
        TH=TH
    )

    final_edges = hysteresis(stage1["gNH"], stage1["gNL"])
    stage1["final"] = final_edges

    return stage1

img = cv2.imread("dom.png", cv2.IMREAD_GRAYSCALE)

img = cv2.imread("dom.png", cv2.IMREAD_GRAYSCALE)

if img is None:
    raise FileNotFoundError("Nie udalo sie wczytac pliku dom.png")

result = canny_full(
    img,
    gauss_ksize=(7, 7),
    gauss_sigma=1.4,
    TL=20,
    TH=60
)

opencv_edges = cv2.Canny(
    result["blur"],
    20,
    60,
    apertureSize=3,
    L2gradient=True
)

plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.imshow(img, cmap="gray")
plt.title("Oryginal")
plt.axis("off")

plt.subplot(2, 2, 2)
plt.imshow(result["gNH"].astype(np.uint8) * 255, cmap="gray")
plt.title("gNH")
plt.axis("off")

plt.subplot(2, 2, 3)
plt.imshow(result["final"], cmap="gray")
plt.title("Moj wynik po histerezie")
plt.axis("off")

plt.subplot(2, 2, 4)
plt.imshow(opencv_edges, cmap="gray")
plt.title("cv2.Canny")
plt.axis("off")

plt.tight_layout()
plt.show()

